# 06. Churn Prediction (ML)

Строим ML-модель для оценки вероятности оттока пользователей: сначала собираем user-level признаки, затем обучаем `churn prediction` модель и смотрим качество.

Основные термины оставляем на английском: `feature engineering`, `churn_label`, `ROC-AUC`, `PR-AUC`, `feature importance`.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import pandas as pd

from src.features import build_user_level_features, make_modeling_dataset
from src.modeling import train_churn_model, get_feature_importance
from src.model_utils import save_artifact

users = pd.read_csv(Path('../data/dim_users.csv'), parse_dates=['registration_ts'])
orders = pd.read_csv(Path('../data/fact_orders.csv'), parse_dates=['order_ts'])
sessions = pd.read_csv(Path('../data/fact_app_sessions.csv'), parse_dates=['session_start_ts', 'session_end_ts'])
ab = pd.read_csv(Path('../data/fact_ab_test_assignments.csv'), parse_dates=['assigned_at'])


In [2]:
reference_ts = orders['order_ts'].max() - pd.Timedelta(days=30)

features = build_user_level_features(
    users=users,
    orders=orders,
    sessions=sessions,
    assignments=ab,
    label_horizon_days=30,
    reference_date=reference_ts,
)
model_cohort = features.loc[features['is_model_eligible'] == 1].copy()
dataset = make_modeling_dataset(model_cohort, eligible_only=False)

target_summary = model_cohort['churn_label'].value_counts(normalize=True).rename('share').to_frame()
target_summary['users'] = model_cohort['churn_label'].value_counts()
target_summary.index = target_summary.index.map({0: 'active_next_30d', 1: 'churn_risk'})
target_summary['share'] = (target_summary['share'] * 100).round(1).astype(str) + '%'
target_summary.style.hide(axis='index')


share,users
83.3%,564
16.7%,113


In [3]:
result = train_churn_model(dataset, target_col='churn_label')

metrics = pd.DataFrame({
    'metric': ['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall', 'TP', 'FP', 'TN', 'FN'],
    'value': [
        f"{result.metrics['roc_auc']:.3f}",
        f"{result.metrics['pr_auc']:.3f}",
        f"{result.metrics['f1']:.3f}",
        f"{result.metrics['precision']:.3f}",
        f"{result.metrics['recall']:.3f}",
        f"{result.metrics['tp']:.0f}",
        f"{result.metrics['fp']:.0f}",
        f"{result.metrics['tn']:.0f}",
        f"{result.metrics['fn']:.0f}",
    ],
})
metrics.style.hide(axis='index')

metric,value
ROC-AUC,0.723
PR-AUC,0.922
F1,0.744
Precision,0.946
Recall,0.613
TP,87
FP,5
TN,23
FN,55


In [4]:
importance = get_feature_importance(result.model, result.feature_names, top_n=20)
importance.head(20)

,feature,importance
0,days_since_last_order,0.081418
1,city_Yekaterinburg,0.056387
2,is_treatment,0.047802
3,avg_discount_rub,0.042293
4,acquisition_channel_social_ads,0.038485
5,city_Kazan,0.037930
6,engagement_ratio,0.035948
7,acquisition_channel_affiliate,0.035249
8,days_since_last_session,0.030321
9,acquisition_channel_paid_search,0.030131


In [5]:
scored = result.x_test.copy()
scored['churn_proba'] = result.y_pred_proba
scored['churn_label_true'] = result.y_test
model_path = save_artifact(result.model, Path('../models/churn_model.joblib'))
importance_path = save_artifact(importance, Path('../models/feature_importance.joblib'))

pd.DataFrame({
    'artifact': ['churn_model', 'feature_importance'],
    'path': [str(model_path), str(importance_path)],
}).style.hide(axis='index')

artifact,path
churn_model,..\models\churn_model.joblib
feature_importance,..\models\feature_importance.joblib
